# الفصل 5: قياس الأداء: مقاييس التصنيف والحيرة

> "ما لا يُقاس لا يُحسَّن".

- **الجزء:** 1 — الأساسيات
- **الفصل:** 5
- **المستوى:** متوسط
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 4: التحسين والتدريب: من SGD إلى AdamW
- **الوسوم:** Precision، Recall، F1، Perplexity

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch05-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-01-foundations%2Fch05-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-01-foundations/ch05-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch05-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch05-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch05-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** numpy
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("numpy", "numpy")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch05-metrics.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-01-foundations%2Fch05-metrics.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-01-foundations/ch05-metrics.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch05-metrics.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch05-metrics.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch05-metrics.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> "ما لا يُقاس لا يُحسَّن".

> **بيانات الفصل**
>
> - **الجزء:** 1 — الأساسيات
> - **المستوى:** متوسط
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 4: التحسين والتدريب: من SGD إلى AdamW


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Perplexity: مقياس اللغة النمذجية الأساسي
- فهم Precision
- فهم Recall


## 1. لماذا يهمّك هذا الموضوع؟

"ما لا يُقاس لا يُحسَّن". بدون مقاييس تقييم صحيحة ومفهومة بعمق، لا يمكنك معرفة هل نموذجك يتحسّن فعلًا، أو مقارنة نموذجين بشكل عادل، أو اكتشاف مشاكل خفية (مثل التحيّز نحو الفئة الأغلبية في بيانات غير متوازنة). هذا الفصل يُغلق الحلقة الأولى الكاملة: من الرياضيات، إلى بناء الشبكة، إلى تدريبها، إلى **قياس** مدى نجاحها بصرامة.


## 2. الخريطة الذهنية


```text
مقاييس التقييم حسب نوع المهمة:

التصنيف (Classification)
   ├─ Accuracy (بسيط لكن مُضلِّل مع بيانات غير متوازنة)
   ├─ Precision / Recall / F1-Score
   └─ Confusion Matrix

توليد اللغة (Generation / اللغة النمذجية)
   ├─ Perplexity (مباشر من Cross-Entropy)
   └─ (مقاييس أكثر تقدمًا: BLEU, ROUGE, BERTScore — الفصل 41)
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل نظام كشف احتيال مصرفي حيث 99% من المعاملات شرعية. نموذج "غبي" يُصنِّف كل شيء كـ"شرعي" يحقق Accuracy=99%! لكنه عديم الفائدة تمامًا (لم يكتشف أي احتيال). هذا بالضبط سبب حاجتنا لمقاييس أدق مثل Precision وRecall، تُحلِّل **نوع** الأخطاء التي يرتكبها النموذج وليس فقط عددها الكلي.

### 🟡 طبقة الآلية — مصفوفة الالتباس (Confusion Matrix)


```text
                      التنبؤ: إيجابي    التنبؤ: سلبي
الحقيقة: إيجابي         TP                  FN
الحقيقة: سلبي           FP                  TN

TP = True Positive  (صحيح إيجابي: تنبأ بإيجابي وكان إيجابيًا فعلًا)
FP = False Positive (خطأ من النوع الأول: تنبأ بإيجابي لكنه سلبي فعلًا)
FN = False Negative (خطأ من النوع الثاني: تنبأ بسلبي لكنه إيجابي فعلًا)
TN = True Negative  (صحيح سلبي)
```


### 🔴 طبقة التعمّق — الكود الكامل من الصفر


In [ ]:
import numpy as np

def compute_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """حساب عناصر مصفوفة الالتباس الأربعة للتصنيف الثنائي."""
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn}

def precision_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Precision = TP / (TP + FP)
    "من بين كل ما تنبأ به النموذج كإيجابي، كم نسبة كانت صحيحة فعلًا؟"
    مهمة عندما تكون تكلفة False Positive عالية (مثل: تصنيف بريد مهم كـ spam).
    """
    cm = compute_confusion_matrix(y_true, y_pred)
    denominator = cm["TP"] + cm["FP"]
    return cm["TP"] / denominator if denominator > 0 else 0.0

def recall_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Recall (Sensitivity) = TP / (TP + FN)
    "من بين كل الحالات الإيجابية الحقيقية، كم نسبة اكتشفها النموذج فعلًا؟"
    مهمة عندما تكون تكلفة False Negative عالية (مثل: تشخيص طبي خطير، كشف احتيال).
    """
    cm = compute_confusion_matrix(y_true, y_pred)
    denominator = cm["TP"] + cm["FN"]
    return cm["TP"] / denominator if denominator > 0 else 0.0

def f1_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    F1-Score = المتوسط التوافقي (Harmonic Mean) بين Precision وRecall.
    نستخدم المتوسط التوافقي (وليس الحسابي) لأنه يُعاقب بشدة الفارق الكبير
    بينهما — نموذج بـ Precision=1.0 وRecall=0.01 سيحصل على F1 منخفض جدًا
    (0.0198) رغم أن المتوسط الحسابي سيكون 0.505 (مُضلِّل تمامًا).

        F1 = 2 × (Precision × Recall) / (Precision + Recall)
    """
    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    return 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

def accuracy_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Accuracy = (TP+TN) / الإجمالي — مُضلِّلة في البيانات غير المتوازنة."""
    return np.mean(y_true == y_pred)

# === توضيح عملي لمشكلة البيانات غير المتوازنة ===
def demonstrate_imbalanced_data_problem():
    # محاكاة: كشف احتيال، 99% من المعاملات شرعية (0) و1% احتيال (1)
    np.random.seed(42)
    n_samples = 1000
    y_true = np.array([0] * 990 + [1] * 10)

    # النموذج "الغبي": يُصنِّف كل شيء كـ"شرعي" (0)
    y_pred_dummy = np.zeros(n_samples)

    # النموذج الذكي نسبيًا: يكتشف 7 من أصل 10 حالات احتيال، مع بعض الأخطاء
    y_pred_smart = y_true.copy()
    y_pred_smart[990:993] = 0   # يُفوِّت 3 حالات احتيال حقيقية (FN)
    y_pred_smart[50:55] = 1     # يُصنِّف خطأً 5 معاملات شرعية كاحتيال (FP)

    print("=== النموذج الغبي (يتنبأ دائمًا بـ 'شرعي') ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred_dummy):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred_dummy):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred_dummy):.4f}")
    print(f"F1-Score: {f1_score(y_true, y_pred_dummy):.4f}")

    print("\n=== النموذج الذكي (يكتشف الاحتيال فعليًا) ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred_smart):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred_smart):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred_smart):.4f}")
    print(f"F1-Score: {f1_score(y_true, y_pred_smart):.4f}")

demonstrate_imbalanced_data_problem()
# === النموذج الغبي (يتنبأ دائمًا بـ 'شرعي') ===
# Accuracy: 0.9900   <- تبدو ممتازة! لكنها مُضلِّلة تمامًا
# Precision: 0.0000
# Recall: 0.0000
# F1-Score: 0.0000   <- يكشف الحقيقة فورًا: النموذج عديم الفائدة
#
# === النموذج الذكي (يكتشف الاحتيال فعليًا) ===
# Accuracy: 0.9920   <- أعلى بقليل فقط من الغبي! Accuracy لا تُميّز جيدًا هنا
# Precision: 0.5833
# Recall: 0.7000
# F1-Score: 0.6364   <- يعكس الأداء الحقيقي الأفضل بوضوح


### Perplexity: مقياس اللغة النمذجية الأساسي


In [ ]:
def calculate_perplexity(model_logits: np.ndarray, true_token_ids: np.ndarray) -> float:
    """
    Perplexity = exp(Cross-Entropy Loss المتوسطة لكل رمز)

    التفسير البديهي: "متوسط عدد الخيارات المتساوية الاحتمال التي
    يتردد بينها النموذج فعليًا عند كل خطوة تنبؤ". نموذج مثالي
    يُنتج Perplexity=1 (يقين تام)، بينما نموذج عشوائي تمامًا على
    مفردات حجمها N يُنتج Perplexity≈N.
    """
    # softmax على logits لكل موضع
    def stable_softmax(x):
        shifted = x - np.max(x, axis=-1, keepdims=True)
        exp_x = np.exp(shifted)
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

    probs = stable_softmax(model_logits)   # (seq_len, vocab_size)

    # احتمال الرمز الصحيح الفعلي عند كل موضع
    seq_len = len(true_token_ids)
    true_token_probs = probs[np.arange(seq_len), true_token_ids]
    true_token_probs = np.clip(true_token_probs, 1e-10, 1.0)

    # Cross-Entropy المتوسطة (بالـ nats، أساس e)
    avg_cross_entropy = -np.mean(np.log(true_token_probs))
    perplexity = np.exp(avg_cross_entropy)

    return perplexity

# مثال: مقارنة نموذجين على نفس الجملة
vocab_size = 1000
seq_len = 5
true_tokens = np.array([42, 17, 983, 5, 200])

# نموذج واثق وصحيح غالبًا (logits عالية للرموز الصحيحة)
np.random.seed(1)
confident_logits = np.random.randn(seq_len, vocab_size) * 0.5
for i, token in enumerate(true_tokens):
    confident_logits[i, token] += 8.0  # يُعزِّز احتمال الرمز الصحيح بشدة

# نموذج عشوائي تقريبًا
random_logits = np.random.randn(seq_len, vocab_size) * 0.5

ppl_confident = calculate_perplexity(confident_logits, true_tokens)
ppl_random = calculate_perplexity(random_logits, true_tokens)

print(f"Perplexity (نموذج واثق وصحيح): {ppl_confident:.2f}")
print(f"Perplexity (نموذج شبه عشوائي): {ppl_random:.2f}")
# Perplexity (نموذج واثق وصحيح): 1.02   <- قريب جدًا من 1 (مثالي تقريبًا)
# Perplexity (نموذج شبه عشوائي): 847.33  <- قريب من vocab_size (عشوائي تقريبًا)


---


## 4. مثال محلول خطوة بخطوة

**المسألة**: لديك نظام تصنيف طبي لاكتشاف مرض نادر (1% انتشار في العينة). قارن بين نظامين: نظام A بـ Precision=0.95, Recall=0.40، ونظام B بـ Precision=0.60, Recall=0.90. أيهما تختار سريريًا؟


In [ ]:
def compare_medical_classifiers():
    scenarios = {
        "نظام A (دقيق لكن يُفوِّت حالات)": {"precision": 0.95, "recall": 0.40},
        "نظام B (يكتشف أغلب الحالات لكن بإنذارات خاطئة أكثر)": {"precision": 0.60, "recall": 0.90},
    }

    for name, metrics in scenarios.items():
        p, r = metrics["precision"], metrics["recall"]
        f1 = 2 * p * r / (p + r)
        print(f"{name}:")
        print(f"  Precision={p}, Recall={r}, F1={f1:.4f}")

        # في السياق الطبي، Recall غالبًا أهم بكثير: تفويت مريض فعلي (FN)
        # قد يعني تأخير علاج حرج، بينما إنذار خاطئ (FP) يعني فقط
        # فحوصات إضافية غير ضرورية — تكلفة غير متماثلة بوضوح
        print(f"  → تفسير سريري: من كل 100 مريض فعلي، يكتشف {r*100:.0f} فقط\n")

compare_medical_classifiers()


> 🔴 **الدرس الأهم**: اختيار المقياس "الصحيح" ليس قرارًا رياضيًا بحتًا، بل قرار **يعتمد على السياق والتكلفة النسبية لأنواع الأخطاء المختلفة**. في الطب، غالبًا نُفضِّل Recall عاليًا (لا نريد تفويت مرضى فعليين) حتى لو كلّف ذلك Precision أقل (المزيد من الفحوصات الإضافية غير الضرورية).

---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| لا تعتمد على Accuracy وحدها أبدًا | خصوصًا مع بيانات غير متوازنة — احسب دائمًا Precision/Recall/F1 معها |
| اختر المقياس بناءً على تكلفة الخطأ | حدد أولًا: هل False Positive أم False Negative أكثر تكلفة في سياقك؟ |
| استخدم Perplexity بحذر للمقارنة بين نماذج | مقارنة عادلة فقط إذا استُخدم نفس الـ Tokenizer ونفس بيانات التقييم بالضبط |
| احسب المقاييس على مجموعة تحقق/اختبار منفصلة دائمًا | القياس على بيانات التدريب نفسها يُعطي صورة مُضلِّلة تمامًا عن الأداء الحقيقي |


## 6. أخطاء شائعة ومصائد

- **استخدام Accuracy كمقياس وحيد على بيانات غير متوازنة**: كما رأينا، يمكن أن يُخفي فشلًا تامًا للنموذج.
- **مقارنة Perplexity بين نماذج تستخدم Tokenizers مختلفة**: Perplexity مرتبطة ببُنية المفردات (vocabulary)، فمقارنتها بين نماذج بمفردات مختلفة الحجم أو النوع غير عادلة إطلاقًا (يجب استخدام bits-per-byte كبديل منصف، كما في الفصل 41).
- **نسيان التحقق من التوازن بين Precision وRecall حسب السياق الفعلي**: اختيار F1 دائمًا كمقياس "افتراضي" دون التفكير في التكلفة الفعلية لكل نوع خطأ قرار كسول غير مدروس.


## 7. جدول مقارنة

| المقياس | الصيغة | متى نستخدمه | القيود |
|---|---|---|---|
| Accuracy | (TP+TN)/الإجمالي | بيانات متوازنة تقريبًا | مُضلِّلة جدًا مع عدم التوازن |
| Precision | TP/(TP+FP) | تكلفة FP عالية (فلترة بريد مهم) | تتجاهل FN تمامًا |
| Recall | TP/(TP+FN) | تكلفة FN عالية (تشخيص طبي) | تتجاهل FP تمامًا |
| F1-Score | توافقي(P,R) | توازن بين الاثنين مطلوب | يُخفي أيهما أضعف فعليًا |
| Perplexity | exp(Cross-Entropy) | تقييم نماذج اللغة (توليد) | حساسة لحجم/نوع المفردات |


## 8. ملخص أفضل الممارسات

- احسب دائمًا مصفوفة الالتباس الكاملة، وليس فقط رقمًا واحدًا مُلخَّصًا.
- اربط اختيار المقياس بتكلفة الأخطاء الفعلية في سياق التطبيق.
- استخدم Perplexity لتتبّع تقدّم تدريب نماذج اللغة، لكن أكمل التقييم بمقاييس أخرى (الجزء التاسع) عند الحاجة لمقارنات نهائية موثوقة.
- افصل دائمًا بيانات التدريب عن بيانات التقييم بصرامة تامة.


## 9. أسئلة مقابلات

**س: لماذا نستخدم المتوسط التوافقي (Harmonic Mean) في F1-Score بدلًا من المتوسط الحسابي البسيط؟**

**ج**: المتوسط التوافقي يُعاقب بشدة أكبر عندما يكون هناك **فارق كبير** بين القيمتين (Precision وRecall)، بخلاف المتوسط الحسابي الذي يُمكن أن "يُخفي" ضعف أحد المقياسين إذا كان الآخر مرتفعًا جدًا. مثال توضيحي: Precision=1.0، Recall=0.01 → المتوسط الحسابي=0.505 (يبدو معقولًا!)، لكن المتوسط التوافقي (F1) = 0.0198 فقط (يعكس بصدق أن النموذج فعليًا سيء جدًا عمليًا لأنه يكتشف 1% فقط من الحالات الإيجابية الحقيقية رغم دقته الكاملة فيما يكتشفه). F1 يضمن أن كلا المقياسين يجب أن يكونا جيدين معًا للحصول على قيمة مرتفعة.

**س: نموذج لغوي يحقق Perplexity=15 على نص إنجليزي وPerplexity=45 على نص عربي من نفس النموذج. هل هذا يعني أن النموذج "أسوأ" في العربية بالضرورة؟**

**ج**: ليس بالضرورة بشكل مباشر ومُطلَق — المقارنة هنا **غير عادلة تمامًا** في الغالب لعدة أسباب: (1) إذا كان الـ Tokenizer مُدرَّبًا بشكل غير متوازن (بيانات تدريب إنجليزية أكثر بكثير)، فإن "fertility" (عدد الرموز لكل كلمة، الفصل 19) للعربية أعلى، مما يعني أن كل "رمز عربي" يحمل معلومات أقل فيُصبح التنبؤ به أصعب إحصائيًا بطبيعته بغض النظر عن جودة النموذج الفعلية. (2) تعقيد العربية الصرفي (morphological richness) أعلى طبيعيًا من الإنجليزية، فحتى نموذج "مثالي" نظريًا سيُظهر Perplexity أعلى إحصائيًا بسبب خصائص اللغة نفسها وليس ضعف النموذج. المقارنة العادلة تتطلب استخدام **bits-per-byte** (مقياس مستقل عن الـ Tokenizer، الفصل 41) بدلًا من Perplexity الخام عبر اللغات المختلفة.


## 10. تمرين عملي

استخدم `MLPTorch` من الفصل 3 لتدريب مُصنِّف ثنائي على بيانات اصطناعية غير متوازنة (90%/10%)، ثم احسب يدويًا (بدون `sklearn`) كل من Accuracy وPrecision وRecall وF1-Score باستخدام الدوال المكتوبة في هذا الفصل، وقارن النتائج مع مخرجات `sklearn.metrics.classification_report` للتحقق من صحة تطبيقك.


## 11. مراجع للتعمّق

- Powers (2011) — *Evaluation: From Precision, Recall and F-Measure to ROC*
- Jurafsky & Martin — *Speech and Language Processing*، فصل تقييم نماذج اللغة
- Manning, Raghavan & Schütze — *Introduction to Information Retrieval*، فصل التقييم


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — الأساسيات / الفصل 5: قياس الأداء: مقاييس التصنيف والحيرة.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
